# 回家作业 —— GeeksforGeeks 教程摘要（OpenAI + 本地 Ollama）

## 练习目标（理念）

把第 1 天的「网页摘要」升级为两条路径：

1. 先用 **OpenAI** 云端模型，对 GeeksforGeeks 教程页做有语气的结构化摘要
2. 再用 **Ollama** 本地开源模型（OpenAI 兼容 `/v1`）做同样任务，对比效果

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 + 清洗 | `requests` + BeautifulSoup，去掉 nav/footer 等噪音 |
| system / user prompts | 控制摘要结构与语气 |
| OpenAI Chat Completions | `openai.chat.completions.create` |
| Ollama 本地模型 | `base_url=http://localhost:11434/v1` |

## 怎么跑

1. 配好 `OPENAI_API_KEY`，先跑云端摘要段落
2. 本机启动 Ollama，并 `ollama pull llama3.2:1b` 后再跑本地段落
3. 示例 URL 指向 GeeksforGeeks 的 Python 教程页，可换成其他教程链接


In [9]:
# ========== 导入：云端 API + 网页解析依赖 ==========

# 导入标准库 os：读环境变量（API Key）
import os
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境变量
from dotenv import load_dotenv
# 从 IPython.display 导入 Markdown/display：在笔记本里渲染摘要
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端：后面既可用于云端，也可指向 Ollama
from openai import OpenAI
# 从 bs4 导入 BeautifulSoup：解析 HTML、剔除无关标签
from bs4 import BeautifulSoup
# 导入 requests：用 HTTP GET 拉取网页
import requests


In [10]:
# ========== 环境：加载并体检 OPENAI_API_KEY ==========

# 加载 .env；override=True 覆盖进程内已有同名变量
load_dotenv(override=True)
# 读取云端密钥
api_key = os.getenv('OPENAI_API_KEY')

# 分支检查密钥是否缺失、前缀是否像 sk-proj-、首尾是否有空白（英文提示保持原样）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [11]:
# ========== 创建默认 OpenAI 客户端（读环境变量里的密钥） ==========
openai = OpenAI()


In [12]:
# ========== system prompt：设定「毒舌但有料」的教程分析人设 ==========
# 可稍后改提示词做实验（例如要求用某种语言输出）；三引号内英文保持原样

system_prompt = """
You are a senior technical content analyst with a sarcastic, witty tone.

You analyze web pages from learning platforms like GeeksforGeeks, especially programming tutorials.

Your job:
- Ignore navigation menus, ads, footer, and unrelated boilerplate text.
- Focus ONLY on actual educational content (tutorials, explanations, examples, headings).
- Identify the structure of the article (sections, topics, subtopics).
- Provide a detailed but engaging summary.

Style:
- Slightly snarky and humorous, but not excessive.
- Still informative and complete.
- Use markdown formatting (headings + bullet points where useful).
- Do NOT wrap output in code blocks.
"""


In [13]:
# ========== user prompt 前缀：规定摘要任务步骤与约束 ==========
# 真正的网页正文会在后面拼到这段英文说明之后

user_prompt_prefix = """
You are given raw text extracted from a GeeksforGeeks page.

Task:
1. Identify the topic of the page.
2. Extract key sections covered in the tutorial.
3. Summarize each section briefly but clearly.
4. Mention practical concepts (code, DS, algorithms, etc.).
5. End with a short witty one-liner summary.

Important:
- Ignore navigation, ads, unrelated links, and repeated headers.
- Do not copy text verbatim.
- Be structured and moderately detailed (not one paragraph).

"""


In [14]:
# ========== 爬虫：拉取 GeeksforGeeks 教程页并清洗正文 ==========

# HEADERS：伪装常见浏览器 User-Agent，降低被站点直接拒识的概率
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/117.0.0.0 Safari/537.36"
    )
}


def fetch_website_contents(url, max_chars=3000):
    """
    Fetch and clean webpage text content.
    """

    try:
        # GET 网页；timeout=10 避免无限挂起
        response = requests.get(url, headers=HEADERS, timeout=10)
        # 非 2xx 状态码直接抛异常，进入 except
        response.raise_for_status()

        # 用 html.parser 把字节流解析成可查询的 DOM 树
        soup = BeautifulSoup(response.content, "html.parser")

        # 提取 <title>；没有标题就用占位英文
        title = soup.title.string.strip() if soup.title else "No title"

        # 删除脚本/样式/导航/页脚等噪音标签（decompose = 从树里移除）
        for tag in soup([
            "script",
            "style",
            "img",
            "svg",
            "noscript",
            "footer",
            "header",
            "nav",
            "aside",
            "form",
            "button",
        ]):
            tag.decompose()

        # 取出可见文本；用换行分隔，并 strip 每段空白
        text = soup.get_text(separator="\n", strip=True)

        # 去掉空行，得到更干净的正文
        lines = [line.strip() for line in text.splitlines()]
        cleaned_text = "\n".join(line for line in lines if line)

        # 拼成 TITLE/CONTENT 两段，方便模型定位
        final_content = f"TITLE:\n{title}\n\nCONTENT:\n{cleaned_text}"

        # 截断到 max_chars，控制送进模型的上下文长度
        return final_content[:max_chars]

    except requests.exceptions.RequestException as e:
        # 网络/HTTP 错误时返回错误字符串，而不是抛崩整个笔记本
        return f"Error fetching website: {e}"


In [15]:
# ========== 云端路径：拼 messages → 调 OpenAI → 展示 Markdown ==========


def build_messages(website_content):
    """把 system +（前缀与正文拼成的 user）装进 messages 列表。"""
    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            # 前缀说明任务，后面接清洗后的网页文本
            "content": f"{user_prompt_prefix}\n\n{website_content}"
        }
    ]

def summarize(url, model="gpt-4.1-mini"):
    """抓取 url 内容并用指定云端模型生成摘要文本。"""
    # 抓取并清洗网页
    website_content = fetch_website_contents(url)

    # 组装 Chat Completions 所需的 messages
    messages = build_messages(website_content)

    # 调用云端 Chat Completions（非流式）
    response = openai.chat.completions.create(
        model=model,
        messages=messages
    )

    # 返回助手正文
    return response.choices[0].message.content


def display_summary(url):
    """summarize 后在笔记本里用 Markdown 渲染。"""
    summary = summarize(url)
    display(Markdown(summary))


# 示例：对 GeeksforGeeks Python 教程页做一次云端摘要
display_summary("https://www.geeksforgeeks.org/python-programming-language-tutorial/")


# Analysis of the GeeksforGeeks Python Tutorial Page

## 1. Topic of the Page:
This is a comprehensive **Python programming tutorial** from GeeksforGeeks aimed at beginners and intermediate learners. It covers everything from installing Python, writing your first "Hello, World!" program, to advanced concepts like decorators and object-oriented programming (OOP).

---

## 2. Key Sections Covered:
- Introduction and Motivation for Python
- Basics of Python Programming
- Functions in Python
- Data Structures in Python
- Object-Oriented Programming (OOP) Concepts

---

## 3. Section Summaries:

### Introduction and Motivation for Python
- Brief on why Python is popular: simple syntax, readability, rich libraries.
- Python as a high-level, cross-platform language used in various domains: AI, web development, data science.
- Prominent users include Google, Netflix, NASA.
- High demand for Python skills in the job market.
- A simple "Hello, World!" code snippet showcases the language's beginner-friendly nature.

### Basics of Python Programming
- Instructions on installing Python on your system (Windows, Mac, Linux).
- Fundamental concepts such as:
  - Writing your first program.
  - Understanding input/output operations.
  - Variables, data types, keywords, and operators.
  - Control flow: conditional statements and loops.
- Sets the foundation for further learning by clarifying essential syntax and programming logic.

### Functions in Python
- Comprehensive dive into Python's function syntax and usage.
- Understanding parameters, return values, and the scope of variables (local vs global).
- Special function features:
  - Recursion (functions calling themselves).
  - Variable-length arguments (*args and **kwargs).
  - First-class functions meaning you can pass functions as arguments and return them.
  - Lambda expressions for creating anonymous functions on the fly.
  - Built-in higher-order functions: map, filter, reduce which help in functional programming style.
  - Inner functions and decorators to modify behavior of other functions dynamically.

### Data Structures in Python
- Detailed exploration of Python’s built-in data types:
  - Strings, Lists, Tuples, Sets, Dictionaries, Arrays.
  - List comprehension—a Pythonic way to create and manipulate lists efficiently.
- Usage of Python's `collections` module for advanced structures such as:
  - Counters (for counting hashable objects).
  - Heapq (priority queues).
  - Deque (double-ended queues).
  - OrderedDict (dictionaries preserving insertion order).
  - Defaultdict (dictionary with default values for missing keys).
- Bridges basic data types to their practical applications in algorithms and data manipulation.

### Object-Oriented Programming (OOP) Concepts
- Introduction to core OOP principles:
  - Encapsulation: bundling data and methods.
  - Inheritance: reusing and extending classes.
  - Polymorphism: designing functions or methods to work on different types.
  - Abstract classes for blueprint classes.
  - Iterators for traversing collection objects.
- Practical understanding of:
  - Defining classes and creating objects.
  - The role of the `self` keyword as the instance reference.
- Helps build modular, reusable, and scalable Python applications.

---

## 4. Practical Concepts Included:
- Sample Python programs and syntax demonstrations.
- Control structures (if, else, loops).
- Function definitions and advanced functional programming features.
- Use of core and extended data types, including those from the `collections` module.
- Understanding and applying OOP principles for real-world code structuring.
- Mentions relevant libraries and frameworks relevant for fields like AI (TensorFlow, Scikit-learn) and web development (Django, Flask).

---

## 5. Witty One-liner Summary:
If Python were a Swiss Army knife, this tutorial would be your handy instruction manual—minus the blade injuries, but with all the coding cuts you need to slice through any problem.

## 作业升级：改用本地 Ollama

把上面的「网页摘要」从 OpenAI 云端，切换到通过 **Ollama** 在本地跑的开源模型。

下面先确认 Ollama 服务在跑、拉取小模型，再复用同一套抓取逻辑换客户端与 prompts。


In [16]:
# ========== 探活：本机 Ollama 默认端口是否在监听 ==========
# 再次导入 requests（本格可独立运行时也有依赖）
import requests
# GET 根路径；若服务正常，content 通常是 b'Ollama is running'
requests.get("http://localhost:11434").content


b'Ollama is running'

In [17]:
# ========== 拉取本地小模型 llama3.2:1b（需本机已安装 ollama CLI） ==========
!ollama pull llama3.2:1b


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 74701a8c35f6: 100% ▕██████████████████▏ 1.3 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 4f659a1e86d7: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [19]:
# ========== 本地路径专用 prompts：更偏「学习指南」结构 ==========
# 三引号内是发给模型的英文指令/格式说明，保持原样（含既有标注文字）

ollama_system_prompt = """
You are an expert technical educator and content analyst.

Your job is to transform raw tutorial content into a structured, easy-to-understand learning guide.

Rules:

* Ignore navigation menus, ads, footers, sidebars, and unrelated content.
* Focus only on educational material.
* Identify the main topic and key concepts.
* Explain concepts in clear, developer-friendly language.
* Summarize code examples rather than reproducing them.
* Be informative, concise, and well-structured.
* Use markdown formatting.
* Do not wrap the response in code blocks.

Output Format:

# 【注】Topic

# 【注】# Overview

Brief description of the tutorial.

# 【注】# Key Concepts

Important topics covered.

# 【注】# Section Breakdown

Short explanation of major sections.

# 【注】# Practical Takeaways

Real-world applications and best practices.

# 【注】# TL;DR

2-3 sentence summary.
"""

ollama_user_prompt_prefix = """
The following text was extracted from a technical tutorial webpage.

Your task is to create a concise study guide.

Requirements:

Identify the main topic.
Remove website noise and boilerplate content.
Summarize important concepts and sections.
Explain practical use cases.
Highlight tips, best practices, or interview-relevant points.
If algorithms or data structures are mentioned, briefly explain their purpose.

Create a summary detailed enough that someone can understand the tutorial without reading the original page.

Content:
"""


In [20]:
# ========== Ollama 路径：OpenAI 兼容客户端 + 本地模型摘要 ==========

# 再次导入 OpenAI（本格强调：同一个 SDK，换 base_url 即可指向本地）
from openai import OpenAI

# Ollama 的 OpenAI 兼容接口根地址（/v1）
OLLAMA_BASE_URL = "http://localhost:11434/v1"
# api_key 对本地 Ollama 通常任意非空即可；这里沿用占位字符串 "ollama"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

def build_ollama_messages(website_content):
    """组装本地模型用的 system/user messages。"""
    return [
        {
            "role": "system",
            "content": ollama_system_prompt
        },
        {
            "role": "user",
            "content": f"{ollama_user_prompt_prefix}\n\n{website_content}"
        }
    ]

def summarize_ollama(url, model="llama3.2:1b"):
    """抓取网页后，用本地 llama 模型生成学习向摘要。"""
    # 复用前面的清洗抓取函数
    website_content = fetch_website_contents(url)

    # 组装 messages
    messages = build_ollama_messages(website_content)

    # 注意：这里实际传入的 model 字符串写死为 llama3.2:1b（形参 model 未使用，保持原逻辑）
    response = ollama.chat.completions.create(
    model="llama3.2:1b", 
    messages=messages
    )

    # 返回助手正文
    return response.choices[0].message.content


def display_ollama_summary(url):
    """本地摘要 + Markdown 展示。"""
    summary = summarize_ollama(url)
    display(Markdown(summary))


# 示例：同一 GeeksforGeeks 教程页，走本地模型再摘要一次
display_ollama_summary("https://www.geeksforgeeks.org/python-programming-language-tutorial/")


**Python Tutorial - GeeksforGeeks**

**Overview**
===============

Python is a high-level scripting language used for data science, automation, artificial intelligence, web development, and more. It's easy to learn, readable, and has a strong library support, making it a popular choice among developers.

**Key Concepts**
================

* High-level language with clean syntax
* Popular for data science, automation, AI, web development, and more
* Provides libraries and frameworks like Django and Flask for web development and tools like Pandas, TensorFlow, and Scikit-learn for artificial intelligence, machine learning, and data analysis
* Cross-platform and compatible with Windows, Mac, and Linux without major changes

**Section Breakdown**
==================

* Installation: Installing Python on the system
* Basics: Understanding Python fundamentals, variables, operators, keywords, data types, conditional statements, loops, functions
* Data Structures: Strings, lists, tuples, dictionaries, sets, arrays, data structure basics
* OOP Concepts: Object-oriented programming (OOP) principles, classes and objects, inheritance, polymorphism

**Practical Takeaways**
=====================

* Use Python for data science, automation, and web development projects
* Implement basic data structures like lists, tuples, dictionaries, sets, arrays
* Write functions using Python's built-in functions, parameters, return values, variable scope, and more
* Apply OOP principles to build modular, reusable, and scalable code

**TL;DR**
===========

Python is a versatile programming language ideal for data science, automation, artificial intelligence, web development, and more. It provides a strong library support, cross-platform compatibility, and ease of use. By learning Python, you can implement basic data structures, understand OOP concepts, and write functions to accomplish various tasks.

**Important Algorithms and Data Structures**
=====================================================

* Lists: Mutable, ordered append/retrieve operations
* Tuples: Immutable, ordered append/retrieve operations
* Strings: Immobile, unordered string manipulation
* Objects (Dictionaries): Key-value pairs can be mutable or immutable